In [ ]:
# ============================================================
# STUDY 1 — SURVEY ANALYSIS NOTEBOOK
#
#
# Purpose:
#   - Update the unit of inference (RESPONSE ID)
#   - Platform tests: paired, participant-level platform means (t-tests + Wilcoxon robustness + BH-FDR)
#   - Within-platform presenter comparisons (paired t-tests + Wilcoxon robustness + BH-FDR)
#   - Ranking tests: platform mean ranks (paired Wilcoxon + BH-FDR), Friedman omnibus, pairwise Wilcoxon posthocs (BH-FDR within criterion)
#   - Produce LaTeX tables that MATCH the printed DataFrames
#   - Replace MixedLM robustness with stable Participant Fixed Effects (cluster-robust SE)
#   - Reliability diagnostics: within-platform source agreement + variance decomposition (stable FE R^2)
#   - Correlation structure by platform means + optional EFA
# ============================================================

# ----------------------------
# 0) Setup
# ----------------------------
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import wilcoxon, friedmanchisquare
from itertools import combinations
import re

from IPython.display import display

# statsmodels for FDR correction + fixed-effects robustness checks
try:
    import statsmodels.api as sm
    import statsmodels.formula.api as smf
except Exception as e:
    print("statsmodels not available yet. Installing...")
    !pip -q install statsmodels
    import statsmodels.api as sm
    import statsmodels.formula.api as smf

# Multiple testing correction (BH-FDR)
try:
    from statsmodels.stats.multitest import multipletests
except Exception as e:
    print("statsmodels.stats.multitest not available; installing statsmodels should fix.")
    !pip -q install statsmodels
    from statsmodels.stats.multitest import multipletests

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ----------------------------
# 1) Upload + Load CSV
# ----------------------------
from google.colab import files
uploaded = files.upload()
csv_path = list(uploaded.keys())[0]
print("Loaded:", csv_path)

raw = pd.read_csv(csv_path)

# ----------------------------
# 2) CONFIG — update here if your column names differ
# ----------------------------
ID_COL = "ResponseId"

SOURCES = ["Ryan", "Max", "CH9", "KFOR"]
DIMENSIONS = ["Objective", "Accurate", "Trustworthy", "Credible", "Legitimate", "LegitPresenter"]

# Qualtrics ranking criteria names in your file (based on your prints)
RANK_CRITERIA = ["Clarity", "E of V", "UseSafety"]

# Mapping from Qualtrics suffix to source for ranking columns:
# Rank - Clarity_1 = Ryan, _2 = CH9, _3 = Max, _4 = KFOR
RANK_SUFFIX_TO_SOURCE = {"_1": "Ryan", "_2": "CH9", "_3": "Max", "_4": "KFOR"}
PLATFORM_OF_SOURCE = {"Ryan": "YouTube", "Max": "YouTube", "CH9": "TV", "KFOR": "TV"}

# Fix known naming oddity if present
if "KFOR - LegitPresenter_1" in raw.columns and "KFOR- LegitPresenter_1" not in raw.columns:
    raw = raw.rename(columns={"KFOR - LegitPresenter_1": "KFOR- LegitPresenter_1"})

def rating_col(src, dim):
    # special case for KFOR presenter legitimacy column name
    if src == "KFOR" and dim == "LegitPresenter" and "KFOR- LegitPresenter_1" in raw.columns:
        return "KFOR- LegitPresenter_1"
    return f"{src} - {dim}_1"

# ----------------------------
# 3) Robust cleaning: drop Qualtrics header/meta rows correctly
#    Definition: a real respondent row has ANY numeric rating in the 24 rating cells.
#    Participant identity = ResponseId
# ----------------------------
rating_cols = []
for s in SOURCES:
    for d in DIMENSIONS:
        c = rating_col(s, d)
        if c in raw.columns:
            rating_cols.append(c)

if len(rating_cols) == 0:
    raise ValueError("No rating columns detected. Check your column naming pattern in rating_col().")

df = raw.copy()

# Convert rating columns to numeric (header rows become NaN)
for c in rating_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Keep rows with any rating present
df = df.loc[df[rating_cols].notna().any(axis=1)].copy()

# Drop rows missing ResponseId (should not happen in real responses)
df = df.dropna(subset=[ID_COL]).copy()

print("Rows after cleaning:", df.shape[0], " Unique participants:", df[ID_COL].nunique())
print("Rating columns found:", len(rating_cols))

# ----------------------------
# 4) Helper functions: stats + LaTeX formatting
# ----------------------------
def bh_fdr(pvals, alpha=0.05):
    """Return adjusted p-values using Benjamini-Hochberg FDR."""
    pvals = np.asarray(pvals, dtype=float)
    reject, p_adj, _, _ = multipletests(pvals, alpha=alpha, method="fdr_bh")
    return p_adj, reject

def fmt_p(p):
    if pd.isna(p):
        return "NA"
    if p < 0.001:
        return "<.001"
    return f"{p:.3f}"

def mean_sd(x):
    x = np.asarray(x, dtype=float)
    return np.nanmean(x), np.nanstd(x, ddof=1)

def paired_dz(a, b):
    """Cohen's dz for paired design: mean(diff)/sd(diff)."""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    diff = b - a
    diff = diff[np.isfinite(diff)]
    if len(diff) < 2:
        return np.nan
    sd = np.std(diff, ddof=1)
    if sd == 0:
        return 0.0
    return np.mean(diff) / sd

def latex_escape(s):
    """Escape common LaTeX special chars for safe table printing."""
    s = str(s)
    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    for k, v in replacements.items():
        s = s.replace(k, v)
    return s

def df_to_latex_table(df_in, caption, label, align=None, footnote=None):
    """Print LaTeX table to console (simple tabular)."""
    df_out = df_in.copy()
    if align is None:
        align = "l" + "c" * (df_out.shape[1] - 1)

    print("\\begin{table}[ht]")
    print("\\centering")
    print(f"\\caption{{{caption}}}")
    print(f"\\label{{{label}}}")
    print(f"\\begin{{tabular}}{{{align}}}")
    print("\\hline")

    # header
    headers = [latex_escape(c) for c in df_out.columns]
    print(" & ".join(headers) + " \\\\ \\hline")

    # rows
    for _, row in df_out.iterrows():
        vals = []
        for v in row.values:
            vals.append(latex_escape(v))
        print(" & ".join(vals) + " \\\\")
    print("\\hline")
    print("\\end{tabular}")
    if footnote:
        print(f"\\\\ \\footnotesize{{{footnote}}}")
    print("\\end{table}")
    print()

# ----------------------------
# 5) Construct participant-level platform means (correct unit)
# ----------------------------
# Wide matrix: one row per participant, columns for each source×dimension
wide = df[[ID_COL]].drop_duplicates().copy().reset_index(drop=True)

# Merge in all ratings per source/dim (mean if duplicates)
tmp = df[[ID_COL] + rating_cols].copy()
tmp = tmp.groupby(ID_COL, as_index=False).mean(numeric_only=True)
wide = wide.merge(tmp, on=ID_COL, how="left")

# Build platform means per dimension:
# YouTube = mean(Ryan, Max); TV = mean(CH9, KFOR)
for dim in DIMENSIONS:
    yt_cols = [rating_col("Ryan", dim), rating_col("Max", dim)]
    tv_cols = [rating_col("CH9", dim), rating_col("KFOR", dim)]
    yt_cols = [c for c in yt_cols if c in wide.columns]
    tv_cols = [c for c in tv_cols if c in wide.columns]

    wide[f"YouTube_{dim}"] = wide[yt_cols].mean(axis=1) if len(yt_cols) else np.nan
    wide[f"TV_{dim}"] = wide[tv_cols].mean(axis=1) if len(tv_cols) else np.nan

# ----------------------------
# 6) TABLE 1 — Platform differences by dimension
#     Paired t-tests (participant-level platform means) + Wilcoxon robustness
# ----------------------------
table1_rows = []
pvals_t = []

for dim in DIMENSIONS:
    yt = wide[f"YouTube_{dim}"].to_numpy(dtype=float)
    tv = wide[f"TV_{dim}"].to_numpy(dtype=float)
    mask = np.isfinite(yt) & np.isfinite(tv)
    yt2, tv2 = yt[mask], tv[mask]
    n = len(yt2)

    yt_m, yt_sd = mean_sd(yt2) if n else (np.nan, np.nan)
    tv_m, tv_sd = mean_sd(tv2) if n else (np.nan, np.nan)
    delta = (tv_m - yt_m) if np.isfinite(tv_m) and np.isfinite(yt_m) else np.nan

    if n >= 5:
        t_stat, p_t = stats.ttest_rel(tv2, yt2)
        dz = paired_dz(yt2, tv2)
    else:
        t_stat, p_t, dz = np.nan, np.nan, np.nan

    pvals_t.append(p_t if np.isfinite(p_t) else 1.0)

    table1_rows.append({
        "Dimension": dim,
        "N": str(n),
        "YouTube M (SD)": f"{yt_m:.2f} ({yt_sd:.2f})" if np.isfinite(yt_m) else "NA",
        "TV M (SD)": f"{tv_m:.2f} ({tv_sd:.2f})" if np.isfinite(tv_m) else "NA",
        "Δ (TV−YT)": f"{delta:.2f}" if np.isfinite(delta) else "NA",
        "t": f"{t_stat:.3f}" if np.isfinite(t_stat) else "NA",
        "p": fmt_p(p_t),
        "d_z": f"{dz:.3f}" if np.isfinite(dz) else "NA",
    })

p_adj_t, _ = bh_fdr(pvals_t, alpha=0.05)
for i, adj in enumerate(p_adj_t):
    table1_rows[i]["p_FDR"] = fmt_p(adj)

table1 = pd.DataFrame(table1_rows)[
    ["Dimension", "N", "YouTube M (SD)", "TV M (SD)", "Δ (TV−YT)", "t", "p", "p_FDR", "d_z"]
]

# Wilcoxon robustness (paired)
wilcox_rows = []
pvals_w = []
for dim in DIMENSIONS:
    yt = wide[f"YouTube_{dim}"].to_numpy(dtype=float)
    tv = wide[f"TV_{dim}"].to_numpy(dtype=float)
    mask = np.isfinite(yt) & np.isfinite(tv)
    yt2, tv2 = yt[mask], tv[mask]
    n = len(yt2)

    if n >= 5:
        try:
            W, p_w = wilcoxon(tv2, yt2, alternative="two-sided", zero_method="wilcox")
        except ValueError:
            W, p_w = np.nan, np.nan
    else:
        W, p_w = np.nan, np.nan

    pvals_w.append(p_w if np.isfinite(p_w) else 1.0)
    wilcox_rows.append({"Dimension": dim, "W_wilcox": W, "p_wilcox": p_w})

wilcox_df = pd.DataFrame(wilcox_rows)
p_adj_w, _ = bh_fdr(wilcox_df["p_wilcox"].fillna(1.0).to_numpy(), alpha=0.05)
wilcox_df["p_wilcox_FDR"] = p_adj_w

table1 = table1.merge(
    wilcox_df.assign(
        W_wilcox=wilcox_df["W_wilcox"].map(lambda x: f"{x:.1f}" if np.isfinite(x) else "NA"),
        p_wilcox=wilcox_df["p_wilcox"].map(fmt_p),
        p_wilcox_FDR=wilcox_df["p_wilcox_FDR"].map(fmt_p),
    )[["Dimension", "W_wilcox", "p_wilcox", "p_wilcox_FDR"]],
    on="Dimension", how="left"
)

table1 = table1[[
    "Dimension", "N", "YouTube M (SD)", "TV M (SD)", "Δ (TV−YT)", "t", "p", "p_FDR", "d_z",
    "W_wilcox", "p_wilcox", "p_wilcox_FDR"
]]

print("\n" + "="*80)
print("TABLE 1 (DataFrame view): Platform differences by dimension (paired; unit = participant)")
print("="*80)
display(table1)

print("\n" + "="*80)
print("LaTeX — TABLE 1")
print("="*80)
df_to_latex_table(
    table1.rename(columns={"p_FDR":"p$_{FDR}$", "d_z":"d$_z$", "p_wilcox_FDR":"p$_{FDR}$ (W)"}),
    caption="Platform differences by evaluative dimension (paired tests on participant-level platform means; $\\Delta$ is TV minus YouTube). Wilcoxon signed-rank tests reported as robustness checks.",
    label="tab:study2_table1_platform",
    align="l" + "c" * (table1.shape[1] - 1),
    footnote="BH-FDR applied across dimensions for both t-tests and Wilcoxon tests (q=.05)."
)

# ----------------------------
# 7) TABLE 2 — Within-platform presenter variation
#     Paired t-tests + Wilcoxon robustness (BH-FDR separately within platform family)
# ----------------------------
def paired_presenter_table(dimensions, left_src, right_src):
    rows = []
    pvals_local = []
    for dim in dimensions:
        left = wide[rating_col(left_src, dim)].to_numpy(dtype=float)
        right = wide[rating_col(right_src, dim)].to_numpy(dtype=float)
        mask = np.isfinite(left) & np.isfinite(right)
        left2, right2 = left[mask], right[mask]
        n = len(left2)

        lm, lsd = mean_sd(left2) if n else (np.nan, np.nan)
        rm, rsd = mean_sd(right2) if n else (np.nan, np.nan)
        delta = rm - lm if np.isfinite(rm) and np.isfinite(lm) else np.nan

        if n >= 5:
            _, p = stats.ttest_rel(right2, left2)
        else:
            p = np.nan

        pvals_local.append(p if np.isfinite(p) else 1.0)

        rows.append({
            "Dimension": dim,
            "N": str(n),
            f"{left_src} M(SD)": f"{lm:.2f} ({lsd:.2f})" if np.isfinite(lm) else "NA",
            f"{right_src} M(SD)": f"{rm:.2f} ({rsd:.2f})" if np.isfinite(rm) else "NA",
            f"Δ ({right_src}−{left_src})": f"{delta:.2f}" if np.isfinite(delta) else "NA",
            "p": fmt_p(p)
        })

    p_adj_local, _ = bh_fdr(pvals_local, alpha=0.05)
    for i, adj in enumerate(p_adj_local):
        rows[i]["p_FDR"] = fmt_p(adj)

    out = pd.DataFrame(rows)[["Dimension","N",f"{left_src} M(SD)",f"{right_src} M(SD)",f"Δ ({right_src}−{left_src})","p","p_FDR"]]
    return out

def paired_presenter_wilcox(dimensions, left_src, right_src):
    rows = []
    pvals_local = []
    for dim in dimensions:
        left = wide[rating_col(left_src, dim)].to_numpy(dtype=float)
        right = wide[rating_col(right_src, dim)].to_numpy(dtype=float)
        mask = np.isfinite(left) & np.isfinite(right)
        left2, right2 = left[mask], right[mask]
        n = len(left2)

        if n >= 5:
            try:
                W, p = wilcoxon(right2, left2, alternative="two-sided", zero_method="wilcox")
            except ValueError:
                W, p = np.nan, np.nan
        else:
            W, p = np.nan, np.nan

        pvals_local.append(p if np.isfinite(p) else 1.0)
        rows.append({"Dimension": dim, "W": W, "p_wilcox": p})

    out = pd.DataFrame(rows)
    p_adj, _ = bh_fdr(out["p_wilcox"].fillna(1.0).to_numpy(), alpha=0.05)
    out["p_wilcox_FDR"] = p_adj
    return out

yt_within = paired_presenter_table(DIMENSIONS, "Ryan", "Max")
tv_within = paired_presenter_table(DIMENSIONS, "CH9", "KFOR")

yt_w = paired_presenter_wilcox(DIMENSIONS, "Ryan", "Max").assign(
    W=lambda d: d["W"].map(lambda x: f"{x:.1f}" if np.isfinite(x) else "NA"),
    p_wilcox=lambda d: d["p_wilcox"].map(fmt_p),
    p_wilcox_FDR=lambda d: d["p_wilcox_FDR"].map(fmt_p),
)
tv_w = paired_presenter_wilcox(DIMENSIONS, "CH9", "KFOR").assign(
    W=lambda d: d["W"].map(lambda x: f"{x:.1f}" if np.isfinite(x) else "NA"),
    p_wilcox=lambda d: d["p_wilcox"].map(fmt_p),
    p_wilcox_FDR=lambda d: d["p_wilcox_FDR"].map(fmt_p),
)

yt_within = yt_within.merge(yt_w[["Dimension","W","p_wilcox","p_wilcox_FDR"]], on="Dimension", how="left")
tv_within = tv_within.merge(tv_w[["Dimension","W","p_wilcox","p_wilcox_FDR"]], on="Dimension", how="left")

table2 = pd.DataFrame({
    "Dimension": yt_within["Dimension"],
    "YouTube N": yt_within["N"],
    "Ryan M(SD)": yt_within["Ryan M(SD)"],
    "Max M(SD)": yt_within["Max M(SD)"],
    "Δ (Max−Ryan)": yt_within["Δ (Max−Ryan)"],
    "p (YT)": yt_within["p"],
    "p_FDR (YT)": yt_within["p_FDR"],
    "W (YT)": yt_within["W"],
    "p_wilcox (YT)": yt_within["p_wilcox"],
    "p_wilcox_FDR (YT)": yt_within["p_wilcox_FDR"],

    "TV N": tv_within["N"],
    "CH9 M(SD)": tv_within["CH9 M(SD)"],
    "KFOR M(SD)": tv_within["KFOR M(SD)"],
    "Δ (KFOR−CH9)": tv_within["Δ (KFOR−CH9)"],
    "p (TV)": tv_within["p"],
    "p_FDR (TV)": tv_within["p_FDR"],
    "W (TV)": tv_within["W"],
    "p_wilcox (TV)": tv_within["p_wilcox"],
    "p_wilcox_FDR (TV)": tv_within["p_wilcox_FDR"],
})

print("\n" + "="*80)
print("TABLE 2 (DataFrame view): Within-platform variation (paired within YouTube and within TV)")
print("="*80)
display(table2)

print("\n" + "="*80)
print("LaTeX — TABLE 2")
print("="*80)
df_to_latex_table(
    table2.rename(columns={"p_FDR (YT)":"p$_{FDR}$ (YT)","p_FDR (TV)":"p$_{FDR}$ (TV)"}),
    caption="Within-platform presenter variation (paired tests within YouTube and within TV). $\\Delta$ reflects within-platform presenter differences.",
    label="tab:study2_table2_withinplatform",
    align="l" + "c" * (table2.shape[1] - 1),
    footnote="BH-FDR applied separately within each platform family (six within-YouTube tests; six within-TV tests)."
)

# ----------------------------
# 8) TABLE 3 — Source-level descriptives by dimension
# ----------------------------
table3_rows = []
for dim in DIMENSIONS:
    row = {"Dimension": dim}
    for src in SOURCES:
        col = rating_col(src, dim)
        x = wide[col].to_numpy(dtype=float)
        x = x[np.isfinite(x)]
        m, sd = mean_sd(x) if len(x) else (np.nan, np.nan)
        row[src] = f"{m:.2f} ({sd:.2f})" if np.isfinite(m) else "NA"
    table3_rows.append(row)

table3 = pd.DataFrame(table3_rows)[["Dimension"] + SOURCES]

print("\n" + "="*80)
print("TABLE 3 (DataFrame view): Source-level descriptives by dimension")
print("="*80)
display(table3)

print("\n" + "="*80)
print("LaTeX — TABLE 3")
print("="*80)
df_to_latex_table(
    table3,
    caption="Within-source descriptive patterns by dimension. Cells report mean (SD) for each source.",
    label="tab:study2_table3_sourcepatterns",
    align="l" + "c" * (table3.shape[1] - 1)
)

# ----------------------------
# RELIABILITY A: Within-platform agreement (parallel-form style)
# Correlation between the two YouTube clips and between the two TV clips, per dimension
# ----------------------------
print("\n" + "="*80)
print("STIMULUS SENSITIVITY: Within-platform agreement across two exemplars (Ryan~Max; CH9~KFOR)")
print("="*80)

agree_rows = []
for dim in DIMENSIONS:
    ryan = wide[rating_col("Ryan", dim)].to_numpy(dtype=float)
    maxv = wide[rating_col("Max", dim)].to_numpy(dtype=float)
    ch9  = wide[rating_col("CH9", dim)].to_numpy(dtype=float)
    kfor = wide[rating_col("KFOR", dim)].to_numpy(dtype=float)

    m1 = np.isfinite(ryan) & np.isfinite(maxv)
    m2 = np.isfinite(ch9) & np.isfinite(kfor)

    r_yt = np.corrcoef(ryan[m1], maxv[m1])[0,1] if m1.sum() >= 5 else np.nan
    r_tv = np.corrcoef(ch9[m2],  kfor[m2])[0,1] if m2.sum() >= 5 else np.nan

    agree_rows.append({
        "Dimension": dim,
        "r(Ryan,Max)": r_yt,
        "r(CH9,KFOR)": r_tv,
        "N": int(min(m1.sum(), m2.sum()))
    })

agree_df = pd.DataFrame(agree_rows)
display(agree_df.round(3))

# ----------------------------
# 9) Rankings — mapping + robust stats
#     TABLE 4: Platform ranking performance (paired Wilcoxon on platform mean ranks)
#     TABLE 4b: Friedman omnibus across 4 sources (per criterion)
#     TABLE 4c: Pairwise Wilcoxon posthocs across sources (BH-FDR within criterion)
# ----------------------------
rank_cols = [c for c in raw.columns if isinstance(c, str) and c.startswith("Rank -")]
print("\nRanking columns found:", len(rank_cols))

if len(rank_cols) == 0:
    print("No ranking columns found. Skipping ranking analyses.")
else:
    rank_df = raw[[ID_COL] + rank_cols].copy()
    rank_df = rank_df.dropna(subset=[ID_COL]).copy()
    rank_df = rank_df.groupby(ID_COL, as_index=False).first()

    for c in rank_cols:
        rank_df[c] = pd.to_numeric(rank_df[c], errors="coerce")

    def parse_rank_col(col):
        m = re.match(r"^Rank\s*-\s*(.+?)_(\d+)$", col)
        if not m:
            return None, None
        crit = m.group(1).strip()
        suf = "_" + m.group(2)
        return crit, suf

    def infer_source_from_colname(col):
        c = col.lower()
        if "ryan" in c: return "Ryan"
        if "ch9" in c or "channel 9" in c: return "CH9"
        if "max" in c: return "Max"
        if "kfor" in c: return "KFOR"
        return ""

    crit_to_map = {}
    for c in rank_cols:
        crit, suf = parse_rank_col(c)
        if crit is None:
            continue
        crit_to_map.setdefault(crit, {})
        src = infer_source_from_colname(c)
        if src == "":
            src = RANK_SUFFIX_TO_SOURCE.get(suf, "")
        if src != "":
            crit_to_map[crit][src] = c

    available_criteria = [crit for crit in RANK_CRITERIA if crit in crit_to_map]
    print("Ranking criteria available:", available_criteria)

    for crit in available_criteria:
        missing_src = [s for s in SOURCES if s not in crit_to_map[crit]]
        if missing_src:
            print(f"⚠️ Ranking mapping incomplete for '{crit}'. Missing sources: {missing_src}")
            print("Mapped columns:", crit_to_map[crit])

    # TABLE 4
    table4_rows = []
    pvals_rank = []

    for crit in available_criteria:
        colmap = crit_to_map[crit]
        if not all(s in colmap for s in SOURCES):
            continue

        w = rank_df[[ID_COL, colmap["Ryan"], colmap["Max"], colmap["CH9"], colmap["KFOR"]]].copy()
        w.columns = [ID_COL, "Ryan", "Max", "CH9", "KFOR"]
        w = w.dropna(subset=["Ryan", "Max", "CH9", "KFOR"]).copy()
        n = w.shape[0]

        w["YouTube_mean_rank"] = w[["Ryan", "Max"]].mean(axis=1)
        w["TV_mean_rank"] = w[["CH9", "KFOR"]].mean(axis=1)

        yt = w["YouTube_mean_rank"].to_numpy(dtype=float)
        tv = w["TV_mean_rank"].to_numpy(dtype=float)

        if n >= 5:
            try:
                W, p = wilcoxon(tv, yt, alternative="two-sided", zero_method="wilcox")
            except ValueError:
                W, p = np.nan, np.nan
        else:
            W, p = np.nan, np.nan

        pvals_rank.append(p if np.isfinite(p) else 1.0)

        yt_mean = np.mean(yt) if len(yt) else np.nan
        tv_mean = np.mean(tv) if len(tv) else np.nan
        delta = tv_mean - yt_mean  # negative => TV better (lower rank)

        if np.isfinite(delta):
            winner = "TV" if delta < 0 else ("YouTube" if delta > 0 else "Tie")
        else:
            winner = "NA"

        table4_rows.append({
            "Criterion": crit,
            "N": str(n),
            "YouTube mean rank": f"{yt_mean:.2f}" if np.isfinite(yt_mean) else "NA",
            "TV mean rank": f"{tv_mean:.2f}" if np.isfinite(tv_mean) else "NA",
            "Δ (TV−YT)": f"{delta:.2f}" if np.isfinite(delta) else "NA",
            "W": f"{W:.1f}" if np.isfinite(W) else "NA",
            "p": fmt_p(p),
            "Winner": winner
        })

    table4 = pd.DataFrame(table4_rows)
    if len(table4) > 0:
        p_adj_rank, _ = bh_fdr(np.array(pvals_rank, dtype=float), alpha=0.05)
        table4["p_FDR"] = [fmt_p(x) for x in p_adj_rank]
        table4 = table4[["Criterion","N","YouTube mean rank","TV mean rank","Δ (TV−YT)","W","p","p_FDR","Winner"]]

    print("\n" + "="*80)
    print("TABLE 4 (DataFrame view): Platform ranking performance (paired Wilcoxon on platform mean ranks)")
    print("="*80)
    display(table4)

    # TABLE 4b
    friedman_rows = []
    for crit in available_criteria:
        colmap = crit_to_map[crit]
        if not all(s in colmap for s in SOURCES):
            continue
        w = rank_df[[ID_COL, colmap["Ryan"], colmap["Max"], colmap["CH9"], colmap["KFOR"]]].copy()
        w.columns = [ID_COL, "Ryan", "Max", "CH9", "KFOR"]
        w = w.dropna(subset=["Ryan","Max","CH9","KFOR"]).copy()
        if w.shape[0] < 5:
            continue
        chi2, p = friedmanchisquare(w["Ryan"], w["Max"], w["CH9"], w["KFOR"])
        friedman_rows.append({"Criterion": crit, "N": str(w.shape[0]), "Friedman χ2": f"{chi2:.2f}", "p": fmt_p(p)})

    friedman_table = pd.DataFrame(friedman_rows)

    print("\n" + "="*80)
    print("TABLE 4b: Friedman omnibus across 4 sources (per criterion)")
    print("="*80)
    display(friedman_table)

    # TABLE 4c
    posthoc_rows = []
    for crit in available_criteria:
        colmap = crit_to_map[crit]
        if not all(s in colmap for s in SOURCES):
            continue

        w = rank_df[[ID_COL, colmap["Ryan"], colmap["Max"], colmap["CH9"], colmap["KFOR"]]].copy()
        w.columns = [ID_COL, "Ryan", "Max", "CH9", "KFOR"]
        w = w.dropna(subset=["Ryan","Max","CH9","KFOR"]).copy()
        if w.shape[0] < 10:
            continue

        pvals = []
        pairs = []
        for a, b in combinations(SOURCES, 2):
            try:
                _, p = wilcoxon(w[a].to_numpy(dtype=float), w[b].to_numpy(dtype=float),
                               alternative="two-sided", zero_method="wilcox")
            except ValueError:
                p = np.nan
            pairs.append((a, b))
            pvals.append(p if np.isfinite(p) else 1.0)

        p_adj, _ = bh_fdr(np.array(pvals, dtype=float), alpha=0.05)

        for (a, b), p_raw, p_fdr in zip(pairs, pvals, p_adj):
            posthoc_rows.append({
                "Criterion": crit,
                "A": a,
                "B": b,
                "N": str(w.shape[0]),
                "p": fmt_p(p_raw),
                "p_FDR": fmt_p(p_fdr)
            })

    posthoc_table = pd.DataFrame(posthoc_rows)
    if len(posthoc_table) > 0:
        posthoc_table = posthoc_table[["Criterion","A","B","N","p","p_FDR"]]

    print("\n" + "="*80)
    print("TABLE 4c: Pairwise Wilcoxon posthocs across sources (BH-FDR within criterion)")
    print("="*80)
    display(posthoc_table)

    # LaTeX outputs
    print("\n" + "="*80)
    print("LaTeX — TABLE 4 (Platform ranks)")
    print("="*80)
    if len(table4) > 0:
        df_to_latex_table(
            table4.rename(columns={"p_FDR":"p$_{FDR}$"}),
            caption="Platform ranking performance (lower rank indicates better performance). Paired Wilcoxon tests compare participant-level platform mean ranks; BH-FDR applied across ranking criteria (q=.05).",
            label="tab:study2_table4_rank_platform",
            align="l" + "c" * (table4.shape[1] - 1)
        )

    print("\n" + "="*80)
    print("LaTeX — TABLE 4b (Friedman omnibus)")
    print("="*80)
    if len(friedman_table) > 0:
        df_to_latex_table(
            friedman_table,
            caption="Friedman omnibus tests across four sources for each ranking criterion.",
            label="tab:study2_table4_rank_friedman",
            align="l" + "c" * (friedman_table.shape[1] - 1)
        )

    print("\n" + "="*80)
    print("LaTeX — TABLE 4c (Pairwise Wilcoxon posthocs)")
    print("="*80)
    if len(posthoc_table) > 0:
        df_to_latex_table(
            posthoc_table.rename(columns={"p_FDR":"p$_{FDR}$"}),
            caption="Pairwise Wilcoxon signed-rank posthoc comparisons between sources for each ranking criterion; BH-FDR applied within criterion (q=.05).",
            label="tab:study2_table4_rank_posthoc",
            align="l" + "c" * (posthoc_table.shape[1] - 1)
        )

# ----------------------------
# 10) Build long ratings data (participant × source × dimension)
# ----------------------------
long_rows = []
for _, r in wide.iterrows():
    pid = r[ID_COL]
    for src in SOURCES:
        platform = PLATFORM_OF_SOURCE[src]
        for dim in DIMENSIONS:
            col = rating_col(src, dim)
            val = r.get(col, np.nan)
            if np.isfinite(val):
                long_rows.append({
                    ID_COL: pid,
                    "Source": src,
                    "Platform": platform,
                    "Platform_bin": 1 if platform == "TV" else 0,  # TV - YouTube
                    "Dimension": dim,
                    "Rating": float(val)
                })
long_df = pd.DataFrame(long_rows)

# ----------------------------
# 11) ROBUSTNESS — Participant Fixed Effects (REPLACES MixedLM)
#     11A: per-dimension FE (cluster-robust SE)
#     11B: Platform × Dimension interaction FE (cluster-robust SE)
# ----------------------------
print("\n" + "="*80)
print("ROBUSTNESS 3A: Participant FE (clustered SE) per dimension — Rating ~ Platform + C(Participant)")
print("="*80)

fe_rows = []
pvals_fe = []

for dim in DIMENSIONS:
    ddf = long_df[long_df["Dimension"] == dim].copy()
    if ddf[ID_COL].nunique() < 10:
        continue

    m = smf.ols(f"Rating ~ Platform_bin + C({ID_COL})", data=ddf).fit(
        cov_type="cluster",
        cov_kwds={"groups": ddf[ID_COL]}
    )

    coef = m.params.get("Platform_bin", np.nan)  # TV - YouTube
    pval = m.pvalues.get("Platform_bin", np.nan)

    fe_rows.append({
        "Dimension": dim,
        "Platform_effect (TV−YT)": coef,
        "p": pval,
        "N_participants": ddf[ID_COL].nunique(),
        "N_obs": len(ddf)
    })
    pvals_fe.append(pval if np.isfinite(pval) else 1.0)

fe_df = pd.DataFrame(fe_rows)
if len(fe_df):
    p_adj, _ = bh_fdr(fe_df["p"].fillna(1.0).to_numpy(), alpha=0.05)
    fe_df["p_FDR_BH"] = p_adj
    display(fe_df.assign(
        **{"Platform_effect (TV−YT)": fe_df["Platform_effect (TV−YT)"].map(lambda x: f"{x:.3f}" if np.isfinite(x) else "NA")},
        p=fe_df["p"].map(fmt_p),
        p_FDR_BH=fe_df["p_FDR_BH"].map(fmt_p),
    )[["Dimension","Platform_effect (TV−YT)","p","p_FDR_BH","N_participants","N_obs"]])

print("\n" + "="*80)
print("ROBUSTNESS 3B: Participant FE interaction (clustered SE) — Rating ~ Platform × Dimension + C(Participant)")
print("="*80)

dim_order = ["Accurate","Credible","LegitPresenter","Legitimate","Objective","Trustworthy"]
long_df["Dimension"] = pd.Categorical(long_df["Dimension"], categories=dim_order, ordered=False)

m_int = smf.ols(f"Rating ~ Platform_bin * Dimension + C({ID_COL})", data=long_df).fit(
    cov_type="cluster",
    cov_kwds={"groups": long_df[ID_COL]}
)
keep_terms = ["Platform_bin"] + [c for c in m_int.params.index
              if c.startswith("Platform_bin:")]
rob3b_table = pd.DataFrame({
    "Term": keep_terms,
    "Coef": m_int.params[keep_terms].round(3),
    "SE": m_int.bse[keep_terms].round(3),
    "z": m_int.tvalues[keep_terms].round(3),
    "p": [fmt_p(m_int.pvalues[t]) for t in keep_terms]
}).reset_index(drop=True)

print(f"N_obs={len(long_df)}, N_participants={long_df[ID_COL].nunique()}, "
      f"R²={m_int.rsquared:.3f}")
display(rob3b_table)

# ----------------------------
# TABLE 5 — Within-source Coverage vs Presenter Legitimacy
# ----------------------------
print("\n" + "="*80)
print("TABLE 5: Coverage vs Presenter Legitimacy (within each source)")
print("="*80)

need_cols = []
for src in SOURCES:
    cov_col = rating_col(src, "Legitimate")
    pres_col = rating_col(src, "LegitPresenter")
    need_cols.extend([cov_col, pres_col])

missing = [c for c in need_cols if c not in wide.columns]
if missing:
    print("⚠️ Missing expected columns for Table 5:")
    for c in missing:
        print("  -", c)
    print("Fix column naming before running Table 5.")
else:
    t5_rows = []
    pvals_t5 = []

    for src in SOURCES:
        cov_col = rating_col(src, "Legitimate")
        pres_col = rating_col(src, "LegitPresenter")

        cov = wide[cov_col].to_numpy(dtype=float)
        pres = wide[pres_col].to_numpy(dtype=float)

        mask = np.isfinite(cov) & np.isfinite(pres)
        cov2, pres2 = cov[mask], pres[mask]
        n = len(cov2)

        cov_m, cov_sd = mean_sd(cov2) if n else (np.nan, np.nan)
        pres_m, pres_sd = mean_sd(pres2) if n else (np.nan, np.nan)
        delta = pres_m - cov_m if np.isfinite(pres_m) and np.isfinite(cov_m) else np.nan

        if n >= 5:
            t_stat, p_val = stats.ttest_rel(pres2, cov2)
            dz = paired_dz(cov2, pres2)
            r_val = np.corrcoef(cov2, pres2)[0, 1] if n >= 3 else np.nan
        else:
            t_stat, p_val, dz, r_val = np.nan, np.nan, np.nan, np.nan

        pvals_t5.append(p_val if np.isfinite(p_val) else 1.0)

        t5_rows.append({
            "Source": src,
            "Platform": PLATFORM_OF_SOURCE.get(src, "NA"),
            "Coverage Legit M(SD)": f"{cov_m:.2f} ({cov_sd:.2f})" if np.isfinite(cov_m) else "NA",
            "Presenter Legit M(SD)": f"{pres_m:.2f} ({pres_sd:.2f})" if np.isfinite(pres_m) else "NA",
            "Δ (Pres−Cov)": f"{delta:.2f}" if np.isfinite(delta) else "NA",
            "t": f"{t_stat:.3f}" if np.isfinite(t_stat) else "NA",
            "p": fmt_p(p_val),
            "d_z": f"{dz:.3f}" if np.isfinite(dz) else "NA",
            "r": f"{r_val:.2f}" if np.isfinite(r_val) else "NA",
            "N": str(n)
        })

    p_adj_t5, _ = bh_fdr(np.array(pvals_t5, dtype=float), alpha=0.05)
    for i, adj in enumerate(p_adj_t5):
        t5_rows[i]["p_FDR"] = fmt_p(adj)

    table5 = pd.DataFrame(t5_rows)[
        ["Source","Platform","Coverage Legit M(SD)","Presenter Legit M(SD)","Δ (Pres−Cov)","t","p","p_FDR","d_z","r","N"]
    ]

    print("\nTABLE 5 (DataFrame view):")
    display(table5)

    print("\n" + "="*80)
    print("LaTeX — TABLE 5")
    print("="*80)
    df_to_latex_table(
        table5.rename(columns={"p_FDR":"p$_{FDR}$","d_z":"d$_z$"}),
        caption="Within-source comparison of coverage legitimacy vs presenter legitimacy. Paired tests compare presenter minus coverage ratings within each source; $r$ is the within-source correlation between the two items.",
        label="tab:study2_table5_legit_cov_vs_pres",
        align="l" + "c" * (table5.shape[1] - 1)
    )

# ----------------------------
# RELIABILITY B: Variance decomposition (stable FE R^2 contributions)
# Model comparisons: intercept-only vs +Participant FE vs +Participant FE + Source FE
# ----------------------------
print("\n" + "="*80)
print("VARIANCE DECOMPOSITION: Signal vs noise per dimension")
print("="*80)

rel_rows = []
for dim in DIMENSIONS:
    ddf = long_df[long_df["Dimension"] == dim].copy()

    m0 = smf.ols("Rating ~ 1", data=ddf).fit()
    mP = smf.ols(f"Rating ~ C({ID_COL})", data=ddf).fit()
    mPS = smf.ols(f"Rating ~ C({ID_COL}) + C(Source)", data=ddf).fit()

    r2_P = mP.rsquared
    r2_PS = mPS.rsquared
    r2_source_given_P = max(r2_PS - r2_P, 0)

    rel_rows.append({
        "Dimension": dim,
        "R2_participant": r2_P,
        "R2_source_given_participant": r2_source_given_P,
        "R2_total_(P+S)": r2_PS,
        "N_obs": len(ddf),
        "N_participants": ddf[ID_COL].nunique()
    })

rel_df = pd.DataFrame(rel_rows)
display(rel_df.round(3))

# ----------------------------
# 12) PART IV — Correlation structure by platform (participant-level platform means)
# ----------------------------
print("\n" + "="*80)
print("PART IV: Correlation structure by platform (participant-level platform means)")
print("="*80)

yt_cols = [f"YouTube_{d}" for d in DIMENSIONS]
tv_cols = [f"TV_{d}" for d in DIMENSIONS]

corr_yt = wide[yt_cols].corr()
corr_tv = wide[tv_cols].corr()

def avg_offdiag(corr_mat):
    vals = corr_mat.to_numpy().copy()
    k = vals.shape[0]
    mask = ~np.eye(k, dtype=bool)
    return np.nanmean(vals[mask])

yt_avg = avg_offdiag(corr_yt)
tv_avg = avg_offdiag(corr_tv)

print(f"YouTube avg off-diagonal r: {yt_avg:.4f}")
print(f"TV avg off-diagonal r: {tv_avg:.4f}")
print(f"Difference (TV - YouTube): {(tv_avg - yt_avg):.4f}\n")

print("YouTube correlation matrix:")
display(corr_yt.round(3))
print("TV correlation matrix:")
display(corr_tv.round(3))

# ============================================================
# OPTIONAL: Exploratory Factor Analysis (EFA)
# Uses participant-level mean across ALL four sources per dimension
# ============================================================
print("\n" + "="*80)
print("EFA: Construct Differentiation Across Six Dimensions")
print("="*80)

try:
    from factor_analyzer import FactorAnalyzer
    from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity
except:
    !pip -q install factor_analyzer
    from factor_analyzer import FactorAnalyzer
    from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity

efa_df = wide[[ID_COL]].copy()
for dim in DIMENSIONS:
    cols = [rating_col(src, dim) for src in SOURCES if rating_col(src, dim) in wide.columns]
    efa_df[dim] = wide[cols].mean(axis=1)

X = efa_df[DIMENSIONS].dropna()
print("Participants used in EFA:", X.shape[0])

kmo_all, kmo_model = calculate_kmo(X)
bartlett_chi2, bartlett_p = calculate_bartlett_sphericity(X)

print(f"KMO overall: {kmo_model:.3f}")
print(f"Bartlett’s test χ²: {bartlett_chi2:.2f}, p = {bartlett_p:.5f}")

fa_test = FactorAnalyzer(rotation=None)
fa_test.fit(X)
ev, _ = fa_test.get_eigenvalues()

print("\nEigenvalues:")
for i, eig in enumerate(ev):
    print(f"Factor {i+1}: {eig:.3f}")

fa1 = FactorAnalyzer(n_factors=1, rotation="varimax")
fa1.fit(X)
loadings_1 = pd.DataFrame(fa1.loadings_, index=DIMENSIONS, columns=["Factor1"])
# Reflect signs if dominant factor loads negatively (sign indeterminacy)
if loadings_1["Factor1"].mean() < 0:
    loadings_1["Factor1"] = loadings_1["Factor1"] * -1
print("\n1-Factor Solution Loadings (signs reflected for interpretability):")
display(loadings_1.round(3))

# Reflect factor for interpretability (optional sign flip)
if loadings_1.mean().values[0] < 0:
    loadings_1["Factor1"] = -loadings_1["Factor1"]

#Because factor sign is indeterminate in exploratory factor analysis, loadings were reflected so that higher item scores correspond to positive factor loadings for interpretability.

fa2 = FactorAnalyzer(n_factors=2, rotation="varimax")
fa2.fit(X)
loadings_2 = pd.DataFrame(fa2.loadings_, index=DIMENSIONS, columns=["Factor1","Factor2"])
print("\n2-Factor Solution Loadings:")
display(loadings_2.round(3))

var = fa2.get_factor_variance()
var_exp = pd.DataFrame({
    "SS Loadings": var[0],
    "Proportion Var": var[1],
    "Cumulative Var": var[2]
}, index=["Factor1","Factor2"])
print("\n2-Factor Variance Explained:")
display(var_exp.round(3))

print("\nEFA complete.")

# ----------------------------
# 13) SANITY CHECKS — if these fail, STOP
# ----------------------------
print("\n" + "="*80)
print("SANITY CHECKS (if these fail, STOP and tell me what printed)")
print("="*80)
print("Unique participants:", wide[ID_COL].nunique())

for dim in DIMENSIONS:
    yt = wide[f"YouTube_{dim}"].to_numpy(dtype=float)
    tv = wide[f"TV_{dim}"].to_numpy(dtype=float)
    mask = np.isfinite(yt) & np.isfinite(tv)
    print(f"{dim:>13}: complete paired platform cases = {mask.sum()}")

print("\nDONE. Copy the LaTeX blocks printed above into Overleaf.")

Saving Trust-Credibility-Legitimacy - FINAL (4).csv to Trust-Credibility-Legitimacy - FINAL (4) (5).csv
Loaded: Trust-Credibility-Legitimacy - FINAL (4) (5).csv
Rows after cleaning: 58  Unique participants: 58
Rating columns found: 24

TABLE 1 (DataFrame view): Platform differences by dimension (paired; unit = participant)


,Dimension,N,YouTube M (SD),TV M (SD),Δ (TV−YT),t,p,p_FDR,d_z,W_wilcox,p_wilcox,p_wilcox_FDR
0,Objective,58,3.69 (0.83),3.89 (0.73),0.20,1.719,0.091,0.091,0.226,367.0,0.128,0.128
1,Accurate,58,3.62 (0.69),4.02 (0.65),0.40,4.070,<.001,<.001,0.534,149.0,<.001,<.001
2,Trustworthy,58,3.51 (0.79),3.84 (0.68),0.33,3.106,0.003,0.004,0.408,166.0,0.001,0.002
3,Credible,58,3.54 (0.85),3.97 (0.62),0.43,3.814,<.001,<.001,0.501,158.0,<.001,<.001
4,Legitimate,58,3.42 (0.90),3.97 (0.71),0.54,4.810,<.001,<.001,0.632,134.0,<.001,<.001
5,LegitPresenter,58,3.49 (0.88),4.01 (0.69),0.52,4.536,<.001,<.001,0.596,121.0,<.001,<.001



LaTeX — TABLE 1
\begin{table}[ht]
\centering
\caption{Platform differences by evaluative dimension (paired tests on participant-level platform means; $\Delta$ is TV minus YouTube). Wilcoxon signed-rank tests reported as robustness checks.}
\label{tab:study2_table1_platform}
\begin{tabular}{lccccccccccc}
\hline
Dimension & N & YouTube M (SD) & TV M (SD) & Δ (TV−YT) & t & p & p\$\_\{FDR\}\$ & d\$\_z\$ & W\_wilcox & p\_wilcox & p\$\_\{FDR\}\$ (W) \\ \hline
Objective & 58 & 3.69 (0.83) & 3.89 (0.73) & 0.20 & 1.719 & 0.091 & 0.091 & 0.226 & 367.0 & 0.128 & 0.128 \\
Accurate & 58 & 3.62 (0.69) & 4.02 (0.65) & 0.40 & 4.070 & <.001 & <.001 & 0.534 & 149.0 & <.001 & <.001 \\
Trustworthy & 58 & 3.51 (0.79) & 3.84 (0.68) & 0.33 & 3.106 & 0.003 & 0.004 & 0.408 & 166.0 & 0.001 & 0.002 \\
Credible & 58 & 3.54 (0.85) & 3.97 (0.62) & 0.43 & 3.814 & <.001 & <.001 & 0.501 & 158.0 & <.001 & <.001 \\
Legitimate & 58 & 3.42 (0.90) & 3.97 (0.71) & 0.54 & 4.810 & <.001 & <.001 & 0.632 & 134.0 & <.001 & <.00

,Dimension,YouTube N,Ryan M(SD),Max M(SD),Δ (Max−Ryan),p (YT),p_FDR (YT),W (YT),p_wilcox (YT),p_wilcox_FDR (YT),TV N,CH9 M(SD),KFOR M(SD),Δ (KFOR−CH9),p (TV),p_FDR (TV),W (TV),p_wilcox (TV),p_wilcox_FDR (TV)
0,Objective,58,3.60 (0.97),3.78 (1.06),0.17,0.273,0.273,220.5,0.268,0.268,58,3.66 (1.10),4.12 (0.70),0.47,0.003,0.016,113.0,0.003,0.020
1,Accurate,58,3.38 (0.97),3.86 (0.89),0.48,0.005,0.011,173.5,0.006,0.016,58,3.81 (1.02),4.22 (0.80),0.41,0.017,0.035,183.0,0.026,0.039
2,Trustworthy,58,3.31 (1.08),3.71 (0.96),0.40,0.022,0.027,273.0,0.034,0.041,58,3.66 (1.05),4.02 (0.87),0.36,0.049,0.059,184.5,0.046,0.056
3,Credible,58,3.34 (1.02),3.74 (1.00),0.40,0.008,0.011,177.5,0.011,0.016,58,3.81 (1.12),4.14 (0.71),0.33,0.079,0.079,228.0,0.056,0.056
4,Legitimate,58,3.22 (1.08),3.62 (1.02),0.40,0.008,0.011,212.0,0.009,0.016,58,3.78 (1.01),4.16 (0.87),0.38,0.023,0.035,206.0,0.023,0.039
5,LegitPresenter,58,3.22 (1.04),3.76 (1.05),0.53,<.001,0.004,173.0,<.001,0.005,58,3.79 (1.12),4.22 (0.77),0.43,0.017,0.035,165.0,0.020,0.039



LaTeX — TABLE 2
\begin{table}[ht]
\centering
\caption{Within-platform presenter variation (paired tests within YouTube and within TV). $\Delta$ reflects within-platform presenter differences.}
\label{tab:study2_table2_withinplatform}
\begin{tabular}{lcccccccccccccccccc}
\hline
Dimension & YouTube N & Ryan M(SD) & Max M(SD) & Δ (Max−Ryan) & p (YT) & p\$\_\{FDR\}\$ (YT) & W (YT) & p\_wilcox (YT) & p\_wilcox\_FDR (YT) & TV N & CH9 M(SD) & KFOR M(SD) & Δ (KFOR−CH9) & p (TV) & p\$\_\{FDR\}\$ (TV) & W (TV) & p\_wilcox (TV) & p\_wilcox\_FDR (TV) \\ \hline
Objective & 58 & 3.60 (0.97) & 3.78 (1.06) & 0.17 & 0.273 & 0.273 & 220.5 & 0.268 & 0.268 & 58 & 3.66 (1.10) & 4.12 (0.70) & 0.47 & 0.003 & 0.016 & 113.0 & 0.003 & 0.020 \\
Accurate & 58 & 3.38 (0.97) & 3.86 (0.89) & 0.48 & 0.005 & 0.011 & 173.5 & 0.006 & 0.016 & 58 & 3.81 (1.02) & 4.22 (0.80) & 0.41 & 0.017 & 0.035 & 183.0 & 0.026 & 0.039 \\
Trustworthy & 58 & 3.31 (1.08) & 3.71 (0.96) & 0.40 & 0.022 & 0.027 & 273.0 & 0.034 & 0.041 & 58 & 

...


2-Factor Solution Loadings:


,Factor1,Factor2
Objective,0.870,0.389
Accurate,0.818,0.337
Trustworthy,0.755,0.505
Credible,0.737,0.520
Legitimate,0.379,0.923
LegitPresenter,0.612,0.612



2-Factor Variance Explained:


,SS Loadings,Proportion Var,Cumulative Var
Factor1,3.059,0.510,0.510
Factor2,2.016,0.336,0.846



EFA complete.

SANITY CHECKS (if these fail, STOP and tell me what printed)
Unique participants: 58
    Objective: complete paired platform cases = 58
     Accurate: complete paired platform cases = 58
  Trustworthy: complete paired platform cases = 58
     Credible: complete paired platform cases = 58
   Legitimate: complete paired platform cases = 58
LegitPresenter: complete paired platform cases = 58

DONE. Copy the LaTeX blocks printed above into Overleaf.
